# 02 · Bronce → Plata: limpieza, normalización y calidad

**Componente B · Sprint 2 (ETL)** · Fase CRISP-DM: *preparación de los datos*

Aplica las reglas que salieron del notebook 01 y valida el resultado con **pandera**. Nada se borra en silencio: cada decisión queda como columna de marca o en el reporte de calidad.

| Entrada (bronce) | Salida (plata) |
| --- | --- |
| `crudo/2025/ranking_2025.xlsx` | `plata/ranking_2025.parquet` |
| `crudo/2025/bi_compania.csv` (RUC) | `plata/calidad_ranking_2025.json` |

In [1]:
from pathlib import Path

ETL = Path.cwd().resolve()
if ETL.name == "notebooks":
    ETL = ETL.parent
DATOS = ETL / "datos"
CRUDO = DATOS / "crudo"      # bronce: datos tal como llegan, sin modificar
SCVS = CRUDO / "2025"        # descargas oficiales de la SCVS
WEB = CRUDO / "web"          # páginas capturadas de las webs oficiales
PLATA = DATOS / "plata"      # datos limpios y validados
ORO = DATOS / "oro"          # datos listos para publicar
for carpeta in (WEB, PLATA, ORO):
    carpeta.mkdir(parents=True, exist_ok=True)

In [2]:
import json
import numpy as np
import pandas as pd
import pandera.pandas as pa

pd.set_option("display.max_columns", 40)

## 1. Lectura y nombres de columnas

In [3]:
ARCHIVO = SCVS / "ranking_2025.xlsx"
fecha_corte = str(pd.read_excel(ARCHIVO, engine="calamine", header=None, nrows=1).iloc[0, 0])
fecha_corte = fecha_corte.replace("Fecha de corte - ", "").strip()

NOMBRES = [
    "posicion_2025", "anio_2025", "posicion_2024", "anio_2024", "expediente", "nombre",
    "tipo_compania", "actividad", "region", "provincia", "ciudad", "tamano_scvs", "sector_scvs",
    "empleados", "activo", "patrimonio", "ventas", "utilidad_antes_impuesto",
    "utilidad_ejercicio", "utilidad_neta", "ir_causado", "ingreso_total",
]
df = pd.read_excel(ARCHIVO, engine="calamine", header=1)
df.columns = NOMBRES
filas_crudo = len(df)
print(fecha_corte, filas_crudo)

Thu Oct 22 00:01:13 ECT 2026 154813


## 2. Normalización de texto y CIIU

In [4]:
for c in ["nombre", "tipo_compania", "actividad", "region", "provincia", "ciudad", "sector_scvs"]:
    df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)

partes = df["actividad"].str.split(" - ", n=1)
df["ciiu"] = partes.str[0].str.strip()
df["ciiu_descripcion"] = partes.str[1].str.strip()
df["seccion_ciiu"] = df["ciiu"].str[0]
df["division_ciiu"] = df["ciiu"].str[:3]
df = df.drop(columns=["actividad", "tamano_scvs"])  # tamano_scvs viene vacía (notebook 01)
df[["expediente", "nombre", "ciiu", "seccion_ciiu", "division_ciiu"]].head()

,expediente,nombre,ciiu,seccion_ciiu,division_ciiu
0,32814,TECMASUR ELECTRONICA INTEGRAL CIA. LTDA.,C3314.03,C,C33
1,93041,SISD CIA. LTDA. SEGURIDAD INTERNACIONAL & SIST...,N8010.03,N,N80
2,176037,GRIALNOVA S.A.,G4669.21,G,G46
3,302181,CONSTRUCTORA IDROVO SILVA HABITARTECH COMPAÑIA...,F4100.10,F,F41
4,305653,REFRIGERACION INDUSTRIAL CORNEJO REFRICOR-EC C...,C2819.11,C,C28


## 3. Duplicados

- Filas idénticas: se quita la copia.
- Mismo expediente con filas distintas: se conserva la de mejor posición 2025 (número menor) y se registra cuáles fueron.

In [5]:
identicas = int(df.duplicated().sum())
df = df.drop_duplicates()

repetidos = df[df["expediente"].duplicated(keep=False)]
expedientes_conflicto = sorted(repetidos["expediente"].unique().tolist())
df = df.sort_values(["expediente", "posicion_2025"]).drop_duplicates("expediente", keep="first")
print("idénticas quitadas:", identicas, "| expedientes con conflicto resueltos:", len(expedientes_conflicto))

idénticas quitadas: 116 | expedientes con conflicto resueltos: 10


## 4. Marcas de calidad

In [6]:
df["anio_cero"] = df["anio_2025"] == 0
df["nueva_en_ranking"] = df["anio_2024"] == 0
df["sin_ventas"] = df["ventas"] == 0
df["sin_actividad"] = (df["ventas"] == 0) & (df["activo"] == 0) & (df["ingreso_total"] == 0)
df["patrimonio_negativo"] = df["patrimonio"] < 0
df["revisar_utilidad"] = (df["utilidad_neta"] > df["ingreso_total"]) & (df["utilidad_neta"] > 0)

## 5. Tamaño derivado por ventas (COPCI)

La columna oficial viene vacía. Se usan las ventas anuales, que el COPCI hace prevalecer sobre el personal cuando no coinciden.

In [7]:
UMBRAL_MEDIANA = 1_000_000
UMBRAL_GRANDE = 5_000_000
df["tamano_derivado"] = np.select(
    [df["ventas"] > UMBRAL_GRANDE, df["ventas"] > UMBRAL_MEDIANA],
    ["grande", "mediana"],
    default="micro_pequena",
)
df["tamano_derivado"].value_counts()

tamano_derivado
micro_pequena    140309
mediana           10041
grande             4337
Name: count, dtype: int64

## 6. Alerta de empleados con su causa

El dato de empleados no se corrige (no hay fuente para el valor real); se marca por qué no es confiable:

| Causa | Regla |
| --- | --- |
| `imposible` | más de 100.000 empleados |
| `grande_menos_10` | grande por ventas y menos de 10 empleados |
| `fuera_de_rango_division` | vende más de 20 veces por empleado que la mediana de su división CIIU |

In [8]:
VECES_LIMITE = 20
df["ventas_por_empleado"] = df["ventas"] / df["empleados"]
mediana_div = df[df["empleados"] >= 10].groupby("division_ciiu")["ventas_por_empleado"].median()
df["veces_mediana_division"] = df["ventas_por_empleado"] / df["division_ciiu"].map(mediana_div)

df["alerta_empleados"] = np.select(
    [
        df["empleados"] > 100_000,
        (df["tamano_derivado"] == "grande") & (df["empleados"] < 10),
        df["veces_mediana_division"] > VECES_LIMITE,
    ],
    ["imposible", "grande_menos_10", "fuera_de_rango_division"],
    default="",
)
df["alerta_empleados"].value_counts()

alerta_empleados
                           152668
fuera_de_rango_division      1148
grande_menos_10               869
imposible                       2
Name: count, dtype: int64

In [9]:
df[df["nombre"].isin(["TIENDAS TUTI TTDE S.A.", "CORPORACION EL ROSADO S.A.", "CORPORACION FAVORITA C.A."])][
    ["nombre", "empleados", "ventas", "ventas_por_empleado", "veces_mediana_division", "alerta_empleados"]
]

,nombre,empleados,ventas,ventas_por_empleado,veces_mediana_division,alerta_empleados
30,CORPORACION FAVORITA C.A.,12033,2.690138e+09,2.235634e+05,2.191086,
38,CORPORACION EL ROSADO S.A.,6,1.641395e+09,2.735658e+08,2681.146071,grande_menos_10
79,TIENDAS TUTI TTDE S.A.,8,1.008144e+09,1.260180e+08,1235.069748,grande_menos_10


## 7. RUC desde el directorio de compañías

In [10]:
directorio = pd.read_csv(SCVS / "bi_compania.csv", dtype={"ruc": str})
directorio = directorio[["expediente", "ruc"]].drop_duplicates("expediente")
df = df.merge(directorio.rename(columns={"ruc": "ruc_original"}), on="expediente", how="left")

# El directorio trae RUC con espacios, letras o puntos: solo se acepta un RUC de 13 dígitos.
limpio = df["ruc_original"].str.strip()
df["ruc_valido"] = limpio.str.fullmatch(r"\d{13}").fillna(False).astype(bool)
df["ruc"] = limpio.where(df["ruc_valido"])
print("filas sin RUC en el directorio:", int(df["ruc_original"].isna().sum()))
print("RUC con formato inválido:", int((df["ruc_original"].notna() & ~df["ruc_valido"]).sum()))
df.loc[df["ruc_original"].notna() & ~df["ruc_valido"], ["expediente", "nombre", "ruc_original"]].head(10)

filas sin RUC en el directorio: 0
RUC con formato inválido: 3


,expediente,nombre,ruc_original
140616,744918,TRANSOCT S.A.,099337330001
145139,750870,SIDCAP ASESORÍA INTEGRAL Y CAPACITACIÓN PROFES...,O195123391001
147866,754309,SALUD SOLIDARIA MÉDICAL CENTER S.A.,0993388798..1


## 8. Validación con pandera

In [11]:
esquema = pa.DataFrameSchema(
    {
        "expediente": pa.Column(int, pa.Check.gt(0), unique=True),
        "nombre": pa.Column(str, pa.Check.str_length(min_value=1)),
        "ciiu": pa.Column(str, pa.Check.str_matches(r"^[A-U]\d{4}")),
        "seccion_ciiu": pa.Column(str, pa.Check.isin(list("ABCDEFGHIJKLMNOPQRSTU"))),
        "region": pa.Column(str, pa.Check.isin(["COSTA", "SIERRA", "ORIENTE", "GALAPAGOS"])),
        "provincia": pa.Column(str, nullable=False),
        "empleados": pa.Column(int, pa.Check.ge(1)),
        "activo": pa.Column(float, pa.Check.ge(0)),
        "ventas": pa.Column(float, pa.Check.ge(0)),
        "ingreso_total": pa.Column(float, pa.Check.ge(0)),
        "tamano_derivado": pa.Column(str, pa.Check.isin(["micro_pequena", "mediana", "grande"])),
        "ruc": pa.Column(str, pa.Check.str_matches(r"^\d{13}$"), nullable=True),
    },
    checks=pa.Check(lambda t: t["ingreso_total"] + 1 >= t["ventas"], error="ventas > ingreso_total"),
    strict=False,
)
plata = esquema.validate(df, lazy=True)
plata.shape

(154687, 37)

## 9. Guardar plata y reporte de calidad

In [12]:
plata = plata.sort_values("posicion_2025").reset_index(drop=True)
plata.to_parquet(PLATA / "ranking_2025.parquet", index=False)

marcas = ["anio_cero", "nueva_en_ranking", "sin_ventas", "sin_actividad", "patrimonio_negativo", "revisar_utilidad"]
calidad = {
    "fuente": "crudo/2025/ranking_2025.xlsx",
    "fecha_corte": fecha_corte,
    "filas_crudo": filas_crudo,
    "filas_identicas_quitadas": identicas,
    "expedientes_con_conflicto": expedientes_conflicto,
    "filas_plata": len(plata),
    "marcas": {m: int(plata[m].sum()) for m in marcas},
    "tamano_derivado": plata["tamano_derivado"].value_counts().to_dict(),
    "alerta_empleados": plata["alerta_empleados"].replace("", "sin_alerta").value_counts().to_dict(),
    "sin_ruc": int(plata["ruc_original"].isna().sum()),
    "ruc_formato_invalido": int((plata["ruc_original"].notna() & ~plata["ruc_valido"]).sum()),
    "reglas": {"umbral_mediana_usd": UMBRAL_MEDIANA, "umbral_grande_usd": UMBRAL_GRANDE,
               "veces_mediana_division": VECES_LIMITE},
}
(PLATA / "calidad_ranking_2025.json").write_text(json.dumps(calidad, ensure_ascii=False, indent=2), encoding="utf-8")
calidad

{'fuente': 'crudo/2025/ranking_2025.xlsx',
 'fecha_corte': 'Thu Oct 22 00:01:13 ECT 2026',
 'filas_crudo': 154813,
 'filas_identicas_quitadas': 116,
 'expedientes_con_conflicto': [55002,
  56726,
  102297,
  130500,
  135453,
  176088,
  300724,
  703139,
  709214,
  743120],
 'filas_plata': 154687,
 'marcas': {'anio_cero': 30,
  'nueva_en_ranking': 22548,
  'sin_ventas': 58905,
  'sin_actividad': 4353,
  'patrimonio_negativo': 17488,
  'revisar_utilidad': 203},
 'tamano_derivado': {'micro_pequena': 140309,
  'mediana': 10041,
  'grande': 4337},
 'alerta_empleados': {'sin_alerta': 152668,
  'fuera_de_rango_division': 1148,
  'grande_menos_10': 869,
  'imposible': 2},
 'sin_ruc': 0,
 'ruc_formato_invalido': 3,
 'reglas': {'umbral_mediana_usd': 1000000,
  'umbral_grande_usd': 5000000,
  'veces_mediana_division': 20}}